# Exploratory analysis

In [ ]:
import pandas as pd
import yaml
import geopandas as gpd
from pathlib import Path

# Load config
with open("../config.yml", "r") as f:
    config = yaml.safe_load(f)

## 1. Business Counts — LAD
`Business_counts_IS8_LADs.parquet` — business counts by IS8 sector, size band, and local authority district (2016–2025)

In [ ]:
# Load file
path = Path("..") / config["paths"]["business_counts_lad"]
df_lad = pd.read_parquet(path)

# --- Basic structure ---
print(f"SHAPE: {df_lad.shape[0]:,} rows x {df_lad.shape[1]} columns")
print(f"\nCOLUMNS:\n{list(df_lad.columns)}")
print(f"\nDTYPES:\n{df_lad.dtypes.to_string()}")
print(f"\nMISSING VALUES:\n{df_lad.isnull().sum().to_string()}")

# --- Key field distributions ---
for col in ["YEAR", "SIZE_BAND", "GEOGRAPHY_TYPE", "IS8_SECTOR", "FRONTIER_SECTOR"]:
    if col in df_lad.columns:
        vals = sorted(df_lad[col].dropna().unique())
        print(f"\nUnique {col} ({len(vals)}):\n  {vals}")

# --- Disclosure control ---
print(f"\nOBS_VALUE description:\n{df_lad['OBS_VALUE'].describe()}")
print(f"\nZero values: {(df_lad['OBS_VALUE'] == 0).sum():,}")
print(f"Values below 5: {(df_lad['OBS_VALUE'] < 5).sum():,}")

# --- Sample rows ---
print(f"\nSAMPLE (5 rows):")
df_lad.head()

In [ ]:
# --- Frequency distribution ---
print(df_lad['OBS_VALUE'].value_counts().sort_index().head(20))

`Business_counts_IS8_LADs.parquet`: Each row answers the question: "In this place, in this year, in this industry, of this size — how many businesses were there?"

**Structure**
- 1.25 million rows, 9 columns
- Unit of observation: a combination of year, size band, geography, IS8 sector (and optionally frontier sector), and SIC code — with a business count as the value
- Covers 2016–2025 (10 years)

**Geography**
- Two geography types in the same file: LADs and country-level aggregates — we'll need to filter out countries when doing LAD analysis

**IS8 Sectors**
- 7 of the 8 IS8 sectors are present — Clean Energy Industries is entirely absent
- "Total" appears as a sector value — it's an economy-wide aggregate, not a real sector, and must be filtered out
- Naming conventions differ slightly from the assignment brief (e.g. "Defence sector", "Digital and Technology") — relevant for any string matching later

**Frontier sectors**
- Only 4 of the 7 present IS8 sectors have frontier sub-sector breakdowns: Advanced Manufacturing, Creative Industries, Financial Services, Professional and Business Services
- Defence, Digital and Technology, and Life Sciences only appear at IS8 level
- The FRONTIER_SECTOR nulls are structural, not a data quality problem

**Disclosure control**
- 918,442 rows (73% of the dataset) have OBS_VALUE of zero
- There are no values between 1 and 4 — ONS suppresses these and rounds to zero
- Zeros mean "zero or suppressed (1–4)", not definitively zero

**Key limitation to document**
- Clean Energy Industries is absent entirely
- Sub-zero suppression means small-area counts are unreliable for precise measurement

## 2. Business Counts — MSOA

In [ ]:
# --- Load file ---
path = Path("..") / config["paths"]["business_counts_msoa"]
df_msoa = pd.read_parquet(path)

# --- Basic structure ---
print(f"SHAPE: {df_msoa.shape[0]:,} rows x {df_msoa.shape[1]} columns")
print(f"\nCOLUMNS:\n{list(df_msoa.columns)}")

# --- Key field distributions ---
print(f"\nUnique GEOGRAPHY_TYPE:\n  {sorted(df_msoa['GEOGRAPHY_TYPE'].dropna().unique())}")
print(f"\nUnique YEAR ({df_msoa['YEAR'].nunique()}):\n  {sorted(df_msoa['YEAR'].dropna().unique())}")
print(f"\nTotal unique geographies: {df_msoa['GEOGRAPHY_CODE'].nunique():,}")

# --- Disclosure control ---
print(f"\nOBS_VALUE description:\n{df_msoa['OBS_VALUE'].describe()}")
print(f"\nZero values: {(df_msoa['OBS_VALUE'] == 0).sum():,}")
print(f"Values below 5: {(df_msoa['OBS_VALUE'] < 5).sum():,}")

# --- Sample rows ---
print(f"\nSAMPLE (5 rows):")
df_msoa.head()

In [ ]:
# --- Frequency distribution ---
print(df_msoa['OBS_VALUE'].value_counts().sort_index().head(20))

`Business_counts_IS8_MSOAs.parquet`: Each row answers the question: "In this small area, in this year, in this industry, of this size — how many businesses were there?"

**Structure**
- 6.3 million rows, 9 columns — same structure as the LAD file
- Same columns, same year range (2016–2025), same IS8/frontier sector logic applies

**Geography**
- Two geography systems: 2021 MSOAs (England and Wales) and 2022 Intermediate Zones (Scotland)
- 1,807 unique geographies total
- No country-level aggregates — unlike the LAD file

**Disclosure control**
- 96% of rows are zero — much higher than the LAD file (73%)
- Same pattern: no values between 1 and 4
- Suppression is severe at this geography — most sector/area combinations are genuinely small

**Key implication**
- MSOA level is better suited for mapping than for regression modelling — suppression is too prevalent for reliable statistical analysis
- Strengthens the case for LAD as the primary unit of analysis

## 3. Employee Counts - LAD

In [ ]:
# --- Load file ---
path = Path("..") / config["paths"]["employee_counts_lad"]
df_emp_lad = pd.read_parquet(path)

# --- Basic structure ---
print(f"SHAPE: {df_emp_lad.shape[0]:,} rows x {df_emp_lad.shape[1]} columns")
print(f"\nCOLUMNS:\n{list(df_emp_lad.columns)}")
print(f"\nDTYPES:\n{df_emp_lad.dtypes.to_string()}")
print(f"\nMISSING VALUES:\n{df_emp_lad.isnull().sum().to_string()}")

# --- Key field distributions ---
for col in ["YEAR", "GEOGRAPHY_TYPE", "IS8_SECTOR", "FRONTIER_SECTOR"]:
    if col in df_emp_lad.columns:
        vals = sorted(df_emp_lad[col].dropna().unique())
        print(f"\nUnique {col} ({len(vals)}):\n  {vals}")

print(f"\nTotal unique geographies: {df_emp_lad['GEOGRAPHY_CODE'].nunique():,}")

# --- Disclosure control ---
print(f"\nOBS_VALUE description:\n{df_emp_lad['OBS_VALUE'].describe()}")
print(f"\nZero values: {(df_emp_lad['OBS_VALUE'] == 0).sum():,}")
print(f"Values below 5: {(df_emp_lad['OBS_VALUE'] < 5).sum():,}")

# --- Sample rows ---
print(f"\nSAMPLE (5 rows):")
df_emp_lad.head()

In [ ]:
# --- Frequency distribution ---
print(df_emp_lad['OBS_VALUE'].value_counts().sort_index().head(20))

`Employee_counts_IS8_LADs.parquet`: Each row answers the question: "In this place, in this year, in this industry — how many employees were there?"

**Structure**
- 312,400 rows, 8 columns — significantly fewer rows than business counts LAD
- No `SIZE_BAND` column — key structural difference from business counts
- Year range 2015–2024 (10 years) — one year earlier start, one year earlier end than business counts

**Geography**
- Same as business counts LAD: LADs and country-level aggregates in the same file
- 355 unique geographies

**IS8 / Frontier sectors**
- Same sector structure as business counts — Clean Energy Industries absent, same 4 sectors with frontier breakdowns

**Disclosure control**
- Much less suppression than business counts — median of 40 employees vs 0 for business counts
- 29% zeros vs 73% in business counts
- BRES is a survey estimate, not an administrative count, so different disclosure rules apply

**Key implication**
- Overlap period with business counts is 2016–2023 — this is the usable window when combining both datasets
- Employee counts will likely be more reliable for analysis than business counts given lower suppression rates

## 4. Employee Counts - MSOA

In [ ]:
# --- Load file ---
path = Path("..") / config["paths"]["employee_counts_msoa"]
df_emp_msoa = pd.read_parquet(path)

# --- Basic structure ---
print(f"SHAPE: {df_emp_msoa.shape[0]:,} rows x {df_emp_msoa.shape[1]} columns")
print(f"\nCOLUMNS:\n{list(df_emp_msoa.columns)}")
print(f"\nMISSING VALUES:\n{df_emp_msoa.isnull().sum().to_string()}")

# --- Key field distributions ---
for col in ["YEAR", "GEOGRAPHY_TYPE", "IS8_SECTOR", "FRONTIER_SECTOR"]:
    if col in df_emp_msoa.columns:
        vals = sorted(df_emp_msoa[col].dropna().unique())
        print(f"\nUnique {col} ({len(vals)}):\n  {vals}")

print(f"\nTotal unique geographies: {df_emp_msoa['GEOGRAPHY_CODE'].nunique():,}")

# --- Disclosure control ---
print(f"\nOBS_VALUE description:\n{df_emp_msoa['OBS_VALUE'].describe()}")
print(f"\nZero values: {(df_emp_msoa['OBS_VALUE'] == 0).sum():,}")
print(f"Values below 5: {(df_emp_msoa['OBS_VALUE'] < 5).sum():,}")

# --- Sample rows ---
print(f"\nSAMPLE (5 rows):")
df_emp_msoa.head()

In [ ]:
# --- Frequency distribution ---
print(df_emp_msoa['OBS_VALUE'].value_counts().sort_index().head(20))

`Employee_counts_IS8_MSOAs.parquet`: Each row answers the question: "In this small area, in this year, in this industry — how many employees were there?"

**Structure**
- 1.58 million rows, 8 columns — same structure as employee counts LAD
- No `SIZE_BAND` column — consistent with LAD employee counts
- Year range 2015–2024 — identical to employee counts LAD

**Geography**
- Same as business counts MSOA: 2021 MSOAs (England and Wales) and 2022 Intermediate Zones (Scotland)
- 1,807 unique geographies — identical to business counts MSOA

**IS8 / Frontier sectors**
- Same sector structure as all previous files

**Disclosure control**
- 81% zeros — higher suppression than employee counts LAD (29%) as expected at finer geography
- Mean of 53 employees vs 5,255 at LAD level — consistent with MSOAs being smaller areas

**Key implication**
- Same conclusion as business counts MSOA: better suited for mapping than modelling
- LAD might be the right unit for regression analysis

## 5. SIC_lookup

In [ ]:
# --- Load file ---
path = Path("..") / config["paths"]["sic_lookup"]
df_sic = pd.read_csv(path)

# --- Basic structure ---
print(f"SHAPE: {df_sic.shape[0]:,} rows x {df_sic.shape[1]} columns")
print(f"\nCOLUMNS:\n{list(df_sic.columns)}")
print(f"\nDTYPES:\n{df_sic.dtypes.to_string()}")
print(f"\nMISSING VALUES:\n{df_sic.isnull().sum().to_string()}")

# --- Key field distributions ---
for col in df_sic.columns:
    n = df_sic[col].nunique()
    vals = sorted(df_sic[col].dropna().unique())
    print(f"\nUnique {col} ({n}):\n  {vals[:20]}")  # cap at 20 to avoid clutter

# --- Sample rows ---
print(f"\nSAMPLE (10 rows):")
df_sic.head(10)

`IS-8_SIC_Lookup.csv`: Each row answers the question: "What IS8 sector and frontier sub-sector does this SIC code belong to?"

**Structure**
- 87 rows, 5 columns — a small reference table, not an analytical dataset
- Maps SIC codes to IS8 sectors and frontier sectors, with descriptions
- SIC codes span 2, 3, 4, and 5 digit levels

**Coverage**
- 84 unique SIC codes mapping to 7 IS8 sectors and 14 frontier sectors
- Clean Energy Industries absent — consistent with all previous files
- 57 nulls in Frontier sector — structural, same pattern as parquet files

**Consistency with parquet files**
- All 84 SIC codes match exactly with INDUSTRY_CODE values in the parquet files
- The one unmatched code is "Total" — the economy-wide aggregate, not a real SIC code

**Role in the pipeline**
- Reference/mapping table — used to validate IS8 and frontier sector labels
- Bridge between raw SIC codes and IS8 classification if needed

## 6. ONS Local Indicators

In [ ]:
# --- Load file ---
path = Path("..") / config["paths"]["ons_indicators"]
xl = pd.ExcelFile(path)

# --- Sheet overview ---
print(f"SHEETS ({len(xl.sheet_names)}):\n  {xl.sheet_names}")

# --- Inspect each sheet ---
for sheet in xl.sheet_names:
    df_sheet = xl.parse(sheet)
    print(f"\n{'='*60}")
    print(f"SHEET: '{sheet}'")
    print(f"  Shape: {df_sheet.shape[0]:,} rows x {df_sheet.shape[1]} columns")
    print(f"  Columns: {list(df_sheet.columns)}")
    print(f"  Sample (3 rows):")
    print(df_sheet.head(3).to_string())

`ONS_local_indicators_package.xlsx`: Each sheet answers the question: "What is the value of this local indicator for this area?"

**Structure**
- 54 sheets — 4 metadata/documentation sheets, 50 indicator sheets
- Each indicator sheet has the same basic structure: metadata rows at the top (variable length), data starting around row 6, Area Code as the join key
- One sheet per indicator — not a tidy long-format table, will require parsing and reshaping before use

**Geography**
- Mixed geographic levels within each sheet — LADs, regions, and nations in the same column
- Uses some pre-2023 LAD codes (obsolete codes flagged in Notes column) — boundary mismatch with parquet files, needs resolution
- Join key is `Area Code` — matches `GEOGRAPHY_CODE` in parquet files

**Coverage**
- Mostly England, some indicators extend to GB or UK — coverage varies by sheet and must be checked per indicator
- Single time point per indicator — dates vary by indicator, roughly 2021–2024
- No time series — each indicator is a snapshot

**Key limitations**
- No panel structure — limits dynamic analysis
- Government R&D only 20 rows — likely not available at LAD level
- University/research intensity absent — needs to be sourced separately (HESA)
- Pre-2023 boundary codes need reconciliation with parquet files

In [ ]:
# --- List of variables available ---
df_dict = xl.parse('Data dictionary', header=None)
print(df_dict.iloc[2:, 1].dropna().to_string())

| EEG/S3 Concept | Variables |
|---|---|
| **Human capital** | NVQ level 3+, GCSEs by age 19, Apprenticeship starts, Apprenticeship achievements, FE participation, FE achievements, KS2 attainment, Ofsted rating, Persistent absences, Persistent absences FSM, Persistent absences CLA, Early years comms, Early years literacy, Early years maths |
| **Entrepreneurial discovery** | Births of enterprises, Deaths of enterprises, Active enterprises, High growth enterprises |
| **Knowledge infrastructure** | Government R&D expenditure |
| **Connectivity and accessibility** | Public transport to employer, Drive to employer, Cycle to employer, Gigabit broadband, 4G coverage |
| **Agglomeration and productivity** | GVA per hour, Weekly pay, Employment rate, Unemployment rate, GDHI per head |
| **Market openness** | UK exports, Inward FDI, Outward FDI |
| **Place conditions** | Net additions to housing stock, Population under devolution deal |
| **Wellbeing and health** | Life satisfaction, Happiness, Worthwhile, Anxiety, Female HLE, Male HLE, Smokers, Reception obesity, Year 6 obesity, Adult obesity, Cancer diagnosis, Under 75 mortality rate, Homicide offences |

In [ ]:
# --- Geographical level of dissagregation ---
results = []

for sheet in xl.sheet_names[4:]:  # skip metadata sheets
    df_s = xl.parse(sheet, header=None)
    
    # Extract metric name
    metric = None
    for i in range(min(10, len(df_s))):
        val = str(df_s.iloc[i, 0])
        if 'Metric:' in val or 'Metric' in val:
            metric = str(df_s.iloc[i, 1]) if pd.notna(df_s.iloc[i, 1]) else val
            break
    
    # Extract all geography rows (Geography: row and any continuation rows below it)
    geos = []
    geo_found = False
    for i in range(min(15, len(df_s))):
        val = str(df_s.iloc[i, 0])
        if 'Geography' in val:
            geo_found = True
            g = str(df_s.iloc[i, 1]) if pd.notna(df_s.iloc[i, 1]) else ''
            if g: geos.append(g)
        elif geo_found:
            # continuation rows have NaN in col 0
            if pd.isna(df_s.iloc[i, 0]) or str(df_s.iloc[i, 0]) == 'nan':
                g = str(df_s.iloc[i, 1]) if pd.notna(df_s.iloc[i, 1]) else ''
                if g and g != 'nan': geos.append(g)
            else:
                break  # hit next metadata field

    results.append({
        'Sheet': sheet,
        'Metric': metric,
        'Geographies': ' | '.join(geos) if geos else 'unknown'
    })

df_geo = pd.DataFrame(results)
print(df_geo.to_string(index=False))

## 7. Boundaries

### LAD geojson

In [ ]:
# --- LAD boundaries ---
path = Path("..") / config["paths"]["lad_boundaries"]
gdf_lad = gpd.read_file(path)
print(f"LAD boundaries")
print(f"  Shape: {gdf_lad.shape}")
print(f"  Columns: {list(gdf_lad.columns)}")
print(f"  CRS: {gdf_lad.crs}")
print(f"Sample LAD codes: {gdf_lad['LAD23CD'].head(3).tolist()}")

### MSOA geojson

In [ ]:
path = Path("..") / config["paths"]["msoa_boundaries"]
gdf_msoa = gpd.read_file(path)
print(f"MSOA boundaries")
print(f"  Shape: {gdf_msoa.shape}")
print(f"  Columns: {list(gdf_msoa.columns)}")
print(f"  CRS: {gdf_msoa.crs}")
print(f"  Sample codes: {gdf_msoa.iloc[:3]['MSOA21CD'].tolist()}")

### IZ geojson

In [ ]:
path = Path("..") / config["paths"]["iz_boundaries"]
gdf_iz = gpd.read_file(path)
print(f"Scottish IZ boundaries")
print(f"  Shape: {gdf_iz.shape}")
print(f"  Columns: {list(gdf_iz.columns)}")
print(f"  CRS: {gdf_iz.crs}")
print(f"  Sample codes: {gdf_iz['IZCode'].head(3).tolist()}")

### MSOA and IZ csv

In [ ]:
# MSOA to combined authority
path = Path("..") / config["paths"]["msoa_to_ca"]
df_msoa_ca = pd.read_csv(path)
print(f"MSOA to Combined Authority")
print(f"  Shape: {df_msoa_ca.shape}")
print(f"  Columns: {list(df_msoa_ca.columns)}")
print(df_msoa_ca.head(3).to_string())

# IZ to council area
path = Path("..") / config["paths"]["iz_to_council"]
df_iz_council = pd.read_csv(path)
print(f"\nIZ to Council Area")
print(f"  Shape: {df_iz_council.shape}")
print(f"  Columns: {list(df_iz_council.columns)}")
print(df_iz_council.head(3).to_string())

# MSOA 2011 to 2021 lookup
path = Path("..") / config["paths"]["msoa_lookup"]
df_msoa_lookup = pd.read_csv(path)
print(f"\nMSOA 2011 to 2021 Lookup")
print(f"  Shape: {df_msoa_lookup.shape}")
print(f"  Columns: {list(df_msoa_lookup.columns)}")
print(df_msoa_lookup.head(3).to_string())

**Boundary and Lookup Files — Summary**

**LAD Boundaries** (`UK_Local_Authority_Districts_December_2023_Boundaries_UK_BGC_...geojson`)
- 361 LADs, 2023 boundaries
- Join key: `LAD23CD` — matches `GEOGRAPHY_CODE` in parquet files exactly
- CRS: EPSG:4326

**MSOA Boundaries** (`Middle_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V3_...geojson`)
- 7,264 MSOAs for England and Wales, 2021 boundaries
- Join key: `MSOA21CD`
- CRS: EPSG:4326

**Scottish IZ Boundaries** (`SG_IZ_2022.geojson`)
- 1,334 Intermediate Zones, 2022 boundaries
- Join key: `IZCode`
- CRS: EPSG:4326

**MSOA to Combined Authority Lookup** (`MSOA_to_English_combined_authorities.csv`)
- 1,941 MSOAs mapped to 10 English combined authorities
- Links `MSOA21CD` → `LAD23CD` → `CAUTH23CD`
- Key file for aggregating to city region level in Phase 2

**IZ to Council Area Lookup** (`IZ2022_to_council_area.csv`)
- 1,334 Scottish IZs mapped to council areas
- Join key: `IZ22CD` — fully consistent with `IZCode` in boundary file
- Key file for aggregating Scottish geographies to Glasgow City Region

**MSOA 2011 to 2021 Crosswalk** (`MSOA_(2011)_to_MSOA_(2021)_to_Local_Authority_District_(2022)_Exact_Fit_Lookup_for_EW_(V2).csv`)
- 7,286 rows linking 2011 MSOAs to 2021 MSOAs
- Contains `CHGIND` column flagging boundary changes
- Key file for resolving pre-2023 boundary mismatch in ONS indicators

**Key implication**
- All join keys confirmed and consistent across files
- The one outstanding issue — pre-2023 LAD codes in ONS indicators — is addressable via the MSOA crosswalk and the `CHGIND` flag

In [ ]:
# Compare IS8-level aggregate vs sum of frontier rows for Sheffield, Advanced Manufacturing, 2023
mask_is8 = (
    (df_emp_lad['GEOGRAPHY_NAME'] == 'Sheffield') &
    (df_emp_lad['IS8_SECTOR'] == 'Advanced manufacturing') &
    (df_emp_lad['YEAR'] == 2023) &
    (df_emp_lad['FRONTIER_SECTOR'].isnull())
)

mask_frontier = (
    (df_emp_lad['GEOGRAPHY_NAME'] == 'Sheffield') &
    (df_emp_lad['IS8_SECTOR'] == 'Advanced manufacturing') &
    (df_emp_lad['YEAR'] == 2023) &
    (df_emp_lad['FRONTIER_SECTOR'].notnull())
)

print("IS8-level rows:")
print(df_emp_lad[mask_is8][['INDUSTRY_CODE', 'FRONTIER_SECTOR', 'OBS_VALUE']].to_string())
print(f"\nFrontier-level rows:")
print(df_emp_lad[mask_frontier][['INDUSTRY_CODE', 'FRONTIER_SECTOR', 'OBS_VALUE']].to_string())
print(f"\nSum of IS8-level OBS_VALUE: {df_emp_lad[mask_is8]['OBS_VALUE'].sum()}")
print(f"Sum of frontier-level OBS_VALUE: {df_emp_lad[mask_frontier]['OBS_VALUE'].sum()}")